In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder ,StandardScaler # Import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score

In [ ]:
from pathlib import Path
import os
DATA_DIR = Path(os.environ.get('OBESITY_DATA_DIR', '.'))
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')

In [ ]:
train

In [ ]:
test.head()

In [ ]:
train.info()

In [ ]:
train.shape

In [ ]:
train.columns

In [ ]:
train.isnull().sum()

In [ ]:
train.describe()

In [ ]:
train.duplicated().sum()

# **transform data**

In [ ]:
# The original dataset schema is not documented; infer only an unambiguous train-only target.
target_candidates = [c for c in train.columns if c not in test.columns and c.lower() != 'id']
target = os.environ.get('OBESITY_TARGET_COLUMN')
if not target:
    if len(target_candidates) != 1:
        raise ValueError('Set OBESITY_TARGET_COLUMN to the actual labeled target column')
    target = target_candidates[0]
if target not in train.columns or target in test.columns:
    raise ValueError('Target must exist only in the labeled training dataset')
if train[target].isna().any():
    raise ValueError('Missing training labels must be resolved before modeling')

In [ ]:
feature_cols = [c for c in train.columns if c != target and c.lower() != 'id']
missing = set(feature_cols) - set(test.columns)
if missing:
    raise ValueError(f'Missing test features: {sorted(missing)}')
x = train[feature_cols].copy()
y = train[target].copy()
X_inference = test[feature_cols].copy()

In [ ]:
x.dtypes

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

categorical = x.select_dtypes(exclude='number').columns.tolist()
numeric = x.select_dtypes(include='number').columns.tolist()
preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric),
    ('categorical', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('encode', OneHotEncoder(handle_unknown='ignore'))]), categorical),
], sparse_threshold=0)


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Fit all transformations exclusively on the training partition.
X_train = preprocessor.fit_transform(x_train)
X_val = preprocessor.transform(x_test)
X_test = preprocessor.transform(X_inference)

In [ ]:
assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1]
assert len(X_test) == len(test)

In [ ]:
X_test.shape

In [ ]:
# Baseline chosen from the classifiers originally imported by this notebook.
model = RandomForestClassifier(random_state=42)

In [ ]:
model.fit(X_train, y_train)
validation_predictions = model.predict(X_val)
print('Validation accuracy:', accuracy_score(y_test, validation_predictions))

In [ ]:
data = x.copy()
data.head()

# **Model**

In [ ]:
train

In [ ]:
data.columns


In [ ]:
predictions = pd.DataFrame({target: model.predict(X_test)})
for column in test.columns:
    if column.lower() == 'id':
        predictions.insert(0, column, test[column].to_numpy())
assert len(predictions) == len(test)
predictions.head()